# Synthesize manipulator (acrobot) control laws (unconstrained + constrained)

This notebook drives the MATLAB reach-avoid backstepping synthesis to generate
**both** acrobot control laws, then loads and displays them:

- **unconstrained** (vanilla baseline) -> `controllers/sop_bounded_control_acrobot_unconstrained.py`
- **constrained** (bounded-control SOP result) -> `controllers/sop_bounded_control_acrobot_result.py`

It runs the canonical, unmodified [`matlab/example_manipulator.m`](../matlab/example_manipulator.m)
via `matlab -batch` (reusing `python/matlab_runner.py`). A single MATLAB run produces both
control laws over the 4-state `[x1,x2,x3,x4] = [q1,q2,dq1,dq2]` (the MATLAB 5-state `x5=x1+x2`
aux is substituted back to 4-state before export).

## Prerequisites
- MATLAB on `PATH` with **SOSTOOLS + Mosek** installed.
- Run with the **`rab_mpc`** kernel (provides `sympy`).
- The SOP solve is **slower than dubins** (`samples_num=10000`, full manipulator dynamics).

In [1]:
# -- Repo-root bootstrap -------------------------------------------------------
# Make ../controllers and ../python importable and locate ../matlab, regardless
# of whether this notebook is launched from notebooks/ or the repository root.
import os
import sys

ROOT = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(ROOT, "controllers")) and os.path.isdir(os.path.join(ROOT, "matlab")):
        break
    ROOT = os.path.dirname(ROOT)

for _sub in ("controllers", "python"):
    _p = os.path.join(ROOT, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

MATLAB_DIR = os.path.join(ROOT, "matlab")
CONTROLLERS_DIR = os.path.join(ROOT, "controllers")
print("ROOT            =", ROOT)
print("MATLAB_DIR      =", MATLAB_DIR)
print("CONTROLLERS_DIR =", CONTROLLERS_DIR)
assert os.path.isdir(MATLAB_DIR), f"matlab/ not found under {ROOT}"

ROOT            = /home/jianqiang/Downloads/reach_avoid_backstepping_MPC
MATLAB_DIR      = /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/matlab
CONTROLLERS_DIR = /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/controllers


In [2]:
# -- Locate the MATLAB executable ---------------------------------------------
from matlab_runner import find_matlab, run_one

# Honour PATH first; fall back to the standard install location on this machine.
MATLAB = find_matlab() or find_matlab("/usr/local/bin/matlab")
if MATLAB is None:
    raise RuntimeError(
        "MATLAB executable not found. Install MATLAB (with SOSTOOLS + Mosek) or add "
        "it to PATH, e.g. `export PATH=$PATH:/usr/local/bin`."
    )
print("MATLAB =", MATLAB)

MATLAB = /usr/local/bin/matlab


In [3]:
# -- Run the MATLAB synthesis (this is the multi-minute step) -----------------
# example_manipulator.m writes BOTH controllers into controllers/ in one run.
EXPORTS = {
    "unconstrained": os.path.join(CONTROLLERS_DIR, "sop_bounded_control_acrobot_unconstrained.py"),
    "constrained":   os.path.join(CONTROLLERS_DIR, "sop_bounded_control_acrobot_result.py"),
}

# Snapshot pre-run mtimes so we can confirm the files were (re)written by this run.
_before = {k: (os.path.getmtime(p) if os.path.exists(p) else None) for k, p in EXPORTS.items()}

result = run_one(MATLAB, MATLAB_DIR, "example_manipulator")

print(f"status      = {result['status']}")
print(f"design_s    = {result['design_s']}")
print(f"sop_solve_s = {result['sop_solve_s']}")
print(f"wall_s      = {result['wall_s']:.1f}")
print("-" * 70)
print("MATLAB stdout (tail):")
print("\n".join(result["stdout"].splitlines()[-25:]))

if result["status"] != "ok":
    raise RuntimeError(
        f"MATLAB synthesis failed ({result['status']}): {result['error']}\n"
        f"stderr:\n{result['stderr']}"
    )

status      = ok
design_s    = 0.309653
sop_solve_s = 216.548743
wall_s      = 222.3
----------------------------------------------------------------------
MATLAB stdout (tail):
 
       cpusec: 0.0731
         iter: 22
    feasratio: 1.1564
         dinf: 0
         pinf: 0
       numerr: 0

__TIMING__,solvesop_bounded_control,sop_k1_solve,214.836777
Final Controller Report: 388 Valid Certificate / 446 Total (87.00%)
__TIMING__,solvesop_bounded_control,final_subs,0.172956
__TIMING__,example_manipulator,sop_solve,216.548743
__TIMING__,example_manipulator,bounds_estimation,0.192166
------------------------------------------------------------------------------------
Obtained controller (4-D, x5 substituted) after solving with bounded control inputs:
-(9609600*cos(x1 + x2) - 1929600*cos(x1) - 19219200*cos(x1 + x2)^2 - 3840000*cos(x1)*cos(x2) - 224864640*sin(x1 + x2)^3 - 3690086400*sin(x1 + x2)^6 + 615014400*cos(x1 + x2)*sin(x1 + x2)^3 + 3859200*cos(x1)^2 + 40040*x3^2*cos(x1 + x2)^2 + 4004

In [4]:
# -- Confirm both control-law files were freshly written ----------------------
import datetime as _dt

for _kind, _path in EXPORTS.items():
    assert os.path.exists(_path), f"Expected export missing: {_path}"
    _mtime = os.path.getmtime(_path)
    _fresh = (_before[_kind] is None) or (_mtime > _before[_kind])
    _stamp = _dt.datetime.fromtimestamp(_mtime).strftime("%Y-%m-%d %H:%M:%S")
    print(f"{_kind:13s} {'(new) ' if _fresh else '(STALE)'}  "
          f"{os.path.getsize(_path):6d} B  {_stamp}  {_path}")
    assert _fresh, f"{_path} was not updated by this run (stale)."

unconstrained (new)    19815 B  2026-06-04 21:05:25  /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/controllers/sop_bounded_control_acrobot_unconstrained.py
constrained   (new)    20136 B  2026-06-04 21:09:00  /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/controllers/sop_bounded_control_acrobot_result.py


In [5]:
# -- Record per-stage + grouped (unconstrained/constrained) timing ------------
# example_manipulator.m / solvesop_bounded_control.m emit one
# `__TIMING__,<script>,<phase>,<seconds>` line per stage; parse_timings collects them.
from matlab_runner import parse_timings

# Which controller each stage contributes to. `design` is shared symbolic groundwork
# for both; the unconstrained (vanilla) controller is produced first, the constrained
# bounded-control controller after it.
_GROUPS = {
    "shared":        ["design"],
    "unconstrained": ["vanilla_solve", "uncon_export"],
    "constrained":   ["sampling", "sop_k1_solve", "final_subs", "bounds_estimation", "result_export"],
}
_STAGE_ORDER = [s for stages in _GROUPS.values() for s in stages]

_phases = parse_timings(result["stdout"])             # every __TIMING__ marker
_total = float(_phases.get("total", result.get("wall_s")))   # MATLAB total, else wall clock

# Flat per-stage dict (umbrella 'sop_solve' and 'total' excluded); plus group subtotals.
_stage_timing = {n: _phases[n] for n in _STAGE_ORDER if n in _phases}
for _n, _v in _phases.items():                        # surface any unexpected extra phase
    if _n not in _stage_timing and _n not in ("sop_solve", "total"):
        _stage_timing[_n] = _v
_subtotal = {g: sum(_phases.get(s, 0.0) for s in stages) for g, stages in _GROUPS.items()}

# Grouped timing table.
print(f"{'group':<14} {'stage':<20} {'seconds':>10}")
print("-" * 46)
for _g, _stages in _GROUPS.items():
    for _s in _stages:
        if _s in _phases:
            print(f"{_g:<14} {_s:<20} {_phases[_s]:>10.3f}")
    print(f"{'':<14} {_g + ' subtotal':<20} {_subtotal[_g]:>10.3f}")
    print("-" * 46)
print(f"{'':<14} {'TOTAL':<20} {_total:>10.3f}")


def embed_timings(py_path, stage_timing, groups, subtotal, total_s):
    """Append (idempotently) the timing + grouped subtotals to an exported .py."""
    marker = "# === stage timings (seconds, auto-generated) ==="
    text = open(py_path).read()
    cut = text.find(marker)
    if cut != -1:                                     # drop any previous block first
        text = text[:cut].rstrip() + "\n"
    L = ["", "", marker, "timing = {"]
    L += [f"    {k!r}: {v}," for k, v in stage_timing.items()]
    L += ["}", "timing_groups = {"]
    L += [f"    {g!r}: {stages!r}," for g, stages in groups.items()]
    L += ["}",
          f"timing_shared_s = {subtotal['shared']}",
          f"timing_unconstrained_s = {subtotal['unconstrained']}",
          f"timing_constrained_s = {subtotal['constrained']}",
          f"timing_total_s = {total_s}", ""]
    with open(py_path, "w") as f:
        f.write(text.rstrip() + "\n" + "\n".join(L))


for _kind, _path in EXPORTS.items():
    embed_timings(_path, _stage_timing, _GROUPS, _subtotal, _total)
    print("embedded timing ->", _path)

group          stage                   seconds
----------------------------------------------
shared         design                    0.310
               shared subtotal           0.310
----------------------------------------------
unconstrained  vanilla_solve             1.179
unconstrained  uncon_export              0.114
               unconstrained subtotal      1.293
----------------------------------------------
constrained    sampling                  0.239
constrained    sop_k1_solve            214.837
constrained    final_subs                0.173
constrained    bounds_estimation         0.192
constrained    result_export             0.091
               constrained subtotal    215.532
----------------------------------------------
               TOTAL                   217.373
embedded timing -> /home/jianqiang/Downloads/reach_avoid_backstepping_MPC/controllers/sop_bounded_control_acrobot_unconstrained.py
embedded timing -> /home/jianqiang/Downloads/reach_avoid_backsteppin

In [6]:
# -- Load + display the synthesized control laws ------------------------------
import importlib

import sympy as sp
from IPython.display import display


def load_controller(module_name):
    """Import (or reload) a freshly written controller module from controllers/."""
    importlib.invalidate_caches()
    if module_name in sys.modules:
        return importlib.reload(sys.modules[module_name])
    return importlib.import_module(module_name)


def _param_block(path):
    """Extract the `# name = value` parameter comments from an export.

    Only lines whose comment body starts with an identifier are kept, so the
    auto-generated `# === stage timings ... ===` marker is not picked up.
    """
    out = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s.startswith("# ") and "=" in s:
                body = s[2:].strip()
                name = body.split("=", 1)[0].strip()
                if name[:1].isalpha() or name[:1] == "_":
                    out.append(body)
    return out


def show_controller(title, module_name, path):
    """Print a summary and render u_opt / k1_opt / certificate_opt for one export."""
    mod = load_controller(module_name)
    u = sp.Matrix(mod.u_opt)            # control law in state space (tau1, tau2)
    k1 = sp.Matrix(mod.k1_opt)          # virtual control in output (y) space
    cert = sp.sympify(mod.certificate_opt)
    print("=" * 72)
    print(title)
    print("module :", module_name)
    print("params :", ", ".join(_param_block(path)) or "(none)")
    print("u_opt   free symbols:", u.free_symbols)
    print("k1_opt  free symbols:", k1.free_symbols)
    print("cert    free symbols:", cert.free_symbols)
    print("sizes  : u_opt ~%d chars, k1_opt ~%d chars, certificate ~%d chars"
          % (len(str(u)), len(str(k1)), len(str(cert))))
    if hasattr(mod, "timing_total_s"):
        print("timing : unconstrained %.3fs + constrained %.3fs + shared %.3fs = total %.3fs"
              % (getattr(mod, "timing_unconstrained_s", float("nan")),
                 getattr(mod, "timing_constrained_s", float("nan")),
                 getattr(mod, "timing_shared_s", float("nan")),
                 mod.timing_total_s))
    print("\nu_opt = [tau1; tau2] :")
    display(u)
    print("k1_opt (output-space virtual control) :")
    display(k1)
    print("certificate_opt (certified set is {certificate_opt >= 0}) :")
    display(cert)
    return mod, u, k1, cert


# --- Unconstrained (vanilla baseline) ---
uncon = show_controller(
    "UNCONSTRAINED (vanilla baseline)",
    "sop_bounded_control_acrobot_unconstrained",
    EXPORTS["unconstrained"],
)

UNCONSTRAINED (vanilla baseline)
module : sop_bounded_control_acrobot_unconstrained
params : lb = [-5500;-700], ub = [5500;700], mu_val = 15, ds = 4, dv = 2
u_opt   free symbols: {x2, x1, x3, x4}
k1_opt  free symbols: {y1, y2}
cert    free symbols: {x2, x1, x3, x4}
sizes  : u_opt ~18676 chars, k1_opt ~188 chars, certificate ~743 chars
timing : unconstrained 1.293s + constrained 215.532s + shared 0.310s = total 217.373s

u_opt = [tau1; tau2] :


Matrix([
[(16000*x3**2*sin(x1)**2*cos(x2) + 8040*x3**2*sin(x1)**2 - 32000*x3**2*sin(x1)*sin(x1 + x2) - 16000*x3**2*sin(x1 + x2)**2*cos(x2) - 40040*x3**2*sin(x1 + x2)**2 + 16000*x3**2*cos(x1)**2*cos(x2) + 8040*x3**2*cos(x1)**2 - 32000*x3**2*cos(x1)*cos(x1 + x2) - 16000*x3**2*cos(x2)*cos(x1 + x2)**2 - 40040*x3**2*cos(x1 + x2)**2 - 32000*x3*x4*sin(x1)*sin(x2)*cos(x1 + x2) + 32000*x3*x4*sin(x1)*sin(x1 + x2)*cos(x2) + 16080*x3*x4*sin(x1)*sin(x1 + x2) + 32000*x3*x4*sin(x2)*sin(x1 + x2)*cos(x1) - 32000*x3*x4*sin(x1 + x2)**2*cos(x2) - 80080*x3*x4*sin(x1 + x2)**2 + 32000*x3*x4*cos(x1)*cos(x2)*cos(x1 + x2) + 16080*x3*x4*cos(x1)*cos(x1 + x2) - 32000*x3*x4*cos(x2)*cos(x1 + x2)**2 - 80080*x3*x4*cos(x1 + x2)**2 - 16000*x3*(-0.208356*sin(x1) - 0.208356*sin(x1 + x2) + 0.0223864*cos(x1) + 0.0223864*cos(x1 + x2) + 0.0014657)*sin(x1)**2*cos(x2) - 8040*x3*(-0.208356*sin(x1) - 0.208356*sin(x1 + x2) + 0.0223864*cos(x1) + 0.0223864*cos(x1 + x2) + 0.0014657)*sin(x1)**2 + 32000*x3*(-0.208356*sin(x1) - 0.208356

k1_opt (output-space virtual control) :


Matrix([
[  -0.084257*y1**2 + 0.067397*y1*y2 + 0.21721*y1 + 6.4062*y2**2 - 0.046771*y2 - 0.097385],
[0.0027983*y1**2 - 0.052089*y1*y2 + 0.0014657*y1 + 0.051187*y2**2 + 0.033986*y2 + 4.3737]])

certificate_opt (certified set is {certificate_opt >= 0}) :


4*(4*sin(x1) + 4*sin(x1 + x2))**3/5 - (-2*(4*sin(x1) + 4*sin(x1 + x2))**3 + 16*cos(x1) + 16*cos(x1 + x2) - 8)**2 - (4*x3*sin(x1) + 4*(x3 + x4)*sin(x1 + x2) + 6.4062*(4*sin(x1) + 4*sin(x1 + x2))**2 + (4*sin(x1) + 4*sin(x1 + x2))*(0.269588*cos(x1) + 0.269588*cos(x1 + x2)) - 0.084257*(4*cos(x1) + 4*cos(x1 + x2))**2 - 0.187084*sin(x1) - 0.187084*sin(x1 + x2) + 0.86884*cos(x1) + 0.86884*cos(x1 + x2) - 0.097385)**2/30 - (-4*x3*cos(x1) - 4*(x3 + x4)*cos(x1 + x2) + 0.051187*(4*sin(x1) + 4*sin(x1 + x2))**2 - (4*sin(x1) + 4*sin(x1 + x2))*(0.208356*cos(x1) + 0.208356*cos(x1 + x2)) + 0.0027983*(4*cos(x1) + 4*cos(x1 + x2))**2 + 0.135944*sin(x1) + 0.135944*sin(x1 + x2) + 0.0058628*cos(x1) + 0.0058628*cos(x1 + x2) + 4.3737)**2/30 + 9.99963229107049

In [7]:
# --- Constrained (bounded-control SOP result) ---
con = show_controller(
    "CONSTRAINED (bounded-control SOP result)",
    "sop_bounded_control_acrobot_result",
    EXPORTS["constrained"],
)

CONSTRAINED (bounded-control SOP result)
module : sop_bounded_control_acrobot_result
params : fx_sym = [[x3], [x4], [(10050*((981*cos(x1 + x2))/50 + (2943*cos(x1))/50 - 8*x4*sin(x2)*(x3 + x4) - 8*x3*x4*sin(x2)))/(160000*cos(x2)**2 - 201201) - (50*((981*cos(x1 + x2))/50 + 8*x3**2*sin(x2))*(400*cos(x2) + 201))/(160000*cos(x2)**2 - 201201)], [(100*((981*cos(x1 + x2))/50 + 8*x3**2*sin(x2))*(400*cos(x2) + 601))/(160000*cos(x2)**2 - 201201) - (50*(400*cos(x2) + 201)*((981*cos(x1 + x2))/50 + (2943*cos(x1))/50 - 8*x4**2*sin(x2) - 16*x3*x4*sin(x2)))/(160000*cos(x2)**2 - 201201)]], gx_sym = [[0, 0], [0, 0], [-10050/(160000*cos(x2)**2 - 201201), (50*(400*cos(x2) + 201))/(160000*cos(x2)**2 - 201201)], [(50*(400*cos(x2) + 201))/(160000*cos(x2)**2 - 201201), -(100*(400*cos(x2) + 601))/(160000*cos(x2)**2 - 201201)]], hx_sym = [[4*cos(x1 + x2) + 4*cos(x1)], [4*sin(x1 + x2) + 4*sin(x1)]], x_vars_sym = [[x1], [x2], [x3], [x4]], y_vars_sym = [[y1], [y2]], safe_set_sym = (4*y2**3)/5 - (2*y2**3 - 4*y1 + 8)

sizes  : u_opt ~17908 chars, k1_opt ~170 chars, certificate ~712 chars
timing : unconstrained 1.293s + constrained 215.532s + shared 0.310s = total 217.373s

u_opt = [tau1; tau2] :


Matrix([
[(16000*x3**2*sin(x1)**2*cos(x2) + 8040*x3**2*sin(x1)**2 - 32000*x3**2*sin(x1)*sin(x1 + x2) - 16000*x3**2*sin(x1 + x2)**2*cos(x2) - 40040*x3**2*sin(x1 + x2)**2 + 16000*x3**2*cos(x1)**2*cos(x2) + 8040*x3**2*cos(x1)**2 - 32000*x3**2*cos(x1)*cos(x1 + x2) - 16000*x3**2*cos(x2)*cos(x1 + x2)**2 - 40040*x3**2*cos(x1 + x2)**2 - 32000*x3*x4*sin(x1)*sin(x2)*cos(x1 + x2) + 32000*x3*x4*sin(x1)*sin(x1 + x2)*cos(x2) + 16080*x3*x4*sin(x1)*sin(x1 + x2) + 32000*x3*x4*sin(x2)*sin(x1 + x2)*cos(x1) - 32000*x3*x4*sin(x1 + x2)**2*cos(x2) - 80080*x3*x4*sin(x1 + x2)**2 + 32000*x3*x4*cos(x1)*cos(x2)*cos(x1 + x2) + 16080*x3*x4*cos(x1)*cos(x1 + x2) - 32000*x3*x4*cos(x2)*cos(x1 + x2)**2 - 80080*x3*x4*cos(x1 + x2)**2 - 16000*x3*(-7.32152*sin(x1) - 7.32152*sin(x1 + x2) + 1.0378*cos(x1) + 1.0378*cos(x1 + x2) + 1.9604)*sin(x1)*cos(x1)*cos(x2) - 8040*x3*(-7.32152*sin(x1) - 7.32152*sin(x1 + x2) + 1.0378*cos(x1) + 1.0378*cos(x1 + x2) + 1.9604)*sin(x1)*cos(x1) - 16000*x3*(-7.32152*sin(x1) - 7.32152*sin(x1 + x2) 

k1_opt (output-space virtual control) :


Matrix([
[      0.07491*y1**2 + 1.855*y1*y2 - 2.9238*y1 + 3.5471*y2**2 - 4.1715*y2 + 5.548],
[0.013837*y1**2 - 0.25945*y1*y2 + 0.70756*y1 + 0.91519*y2**2 - 1.9604*y2 + 1.6486]])

certificate_opt (certified set is {certificate_opt >= 0}) :


4*(4*sin(x1) + 4*sin(x1 + x2))**3/5 - (-2*(4*sin(x1) + 4*sin(x1 + x2))**3 + 16*cos(x1) + 16*cos(x1 + x2) - 8)**2 - (4*x3*sin(x1) + 4*(x3 + x4)*sin(x1 + x2) + 3.5471*(4*sin(x1) + 4*sin(x1 + x2))**2 + (4*sin(x1) + 4*sin(x1 + x2))*(7.42*cos(x1) + 7.42*cos(x1 + x2)) + 0.07491*(4*cos(x1) + 4*cos(x1 + x2))**2 - 16.686*sin(x1) - 16.686*sin(x1 + x2) - 11.6952*cos(x1) - 11.6952*cos(x1 + x2) + 5.548)**2/30 - (-4*x3*cos(x1) - 4*(x3 + x4)*cos(x1 + x2) + 0.91519*(4*sin(x1) + 4*sin(x1 + x2))**2 - (4*sin(x1) + 4*sin(x1 + x2))*(1.0378*cos(x1) + 1.0378*cos(x1 + x2)) + 0.013837*(4*cos(x1) + 4*cos(x1 + x2))**2 - 7.8416*sin(x1) - 7.8416*sin(x1 + x2) + 2.83024*cos(x1) + 2.83024*cos(x1 + x2) + 1.6486)**2/30 + 33.822204552369

## Summary

Both control laws come from the **same** MATLAB run of `example_manipulator.m`:

- **Unconstrained** -- the vanilla backstepping `k1` controller solved **without** torque-bound
  enforcement (`solve_vanilla_k1_controller`); the baseline closed-form law.
- **Constrained** -- the bounded-control **SOP** result (`solve_k1_controller_sop`), which enforces
  the torque bounds `lb=[-5500;-700]`, `ub=[5500;700]` N*m over the sampled state set.

Each exported certificate is shifted by `-delta/lambda` (the certified reach-avoid set is
`{certificate_opt >= 0}`), with the solved `delta` substituted numerically, so no free `delta`
symbol remains in the exports.

### Stage timings (split by controller)
Each exported `.py` ends with an auto-generated `timing = {...}` dict (seconds per stage),
a `timing_groups` map, and grouped subtotals `timing_shared_s` / `timing_unconstrained_s` /
`timing_constrained_s` / `timing_total_s`. Stages are grouped as:

- **shared**: `design` (symbolic backstepping, needed by both controllers)
- **unconstrained**: `vanilla_solve`, `uncon_export`
- **constrained**: `sampling`, `sop_k1_solve`, `final_subs`, `bounds_estimation`, `result_export`

They come from `__TIMING__` markers emitted by the MATLAB code and are parsed/embedded by the
timing cell above. (`design` is shared groundwork, so it is reported separately rather than
charged to either controller.)